In [ ]:
#function cleaning data
def basic_cleaning(df):
    df = df.copy()

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Replace infinite values if any
    df = df.replace([np.inf, -np.inf], np.nan)

    return df

def merge_datasets(repositories, versions, attributes):
    """Merge the repositories, versions, and attributes datasets into a single DataFrame."""
    repos = basic_cleaning(repositories)
    vers = basic_cleaning(versions)
    attrs = basic_cleaning(attributes)

    # Merge repositories with versions if a common key exists
    common_keys_rv = list(set(repos.columns).intersection(set(vers.columns)))
    if common_keys_rv:
        key = common_keys_rv[0]
        merged_df = pd.merge(repos, vers, on=key, how="inner")
    else:
        merged_df = repos.copy()

    # Merge with attributes if a common key exists
    common_keys_all = list(set(merged_df.columns).intersection(set(attrs.columns)))
    if common_keys_all:
        key = common_keys_all[0]
        merged_df = pd.merge(merged_df, attrs, on=key, how="inner")

    return merged_df


def handle_missing_values(df):
    """Handle missing values by filling numeric columns with median and categorical columns with mode."""
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    for col in categorical_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mode()[0])

    return df


def save_processed_data(df, filename="cleaned_dataset.csv"):
    """Save the processed DataFrame to a CSV file."""
    output_path = filename
    df.to_csv(output_path, index=False)
    return output_path


def preprocess_pipeline():
    """Run the full preprocessing pipeline: load data, merge, clean, and save."""
    merged_df = merge_datasets(data_repo, data_versions, data_attribute)
    cleaned_df = handle_missing_values(merged_df)
    output_path = save_processed_data(cleaned_df)
    return cleaned_df, output_path


In [ ]:
clean_data, output_path = preprocess_pipeline()

In [ ]:
clean_data.head()
clean_data.describe()